# Customer Churn Prediction System
### Phase 6: Build the Prediction System (Tasks 17-19)

Phase 5 gave us a tuned Random Forest with ~99.8% F1-score, and confirmed
`Payment Delay` and `Support Calls` as the dominant churn drivers. This phase
wraps that model into an actual usable tool: feed it a new customer's info, get
back a prediction, a probability, a risk level, and a concrete recommendation.

## Setup: rebuilding the tuned model from Phase 5

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('customer_churn.csv').drop(columns=['CustomerID'])
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_prep = X_train.copy()
gender_map = {'Male': 0, 'Female': 1}
X_train_prep['Gender'] = X_train_prep['Gender'].map(gender_map)

categorical_cols = ['Subscription Type', 'Contract Length']
X_train_prep = pd.get_dummies(X_train_prep, columns=categorical_cols, drop_first=True)

numeric_cols = ['Age', 'Tenure', 'Usage Frequency', 'Support Calls',
                'Payment Delay', 'Total Spend', 'Last Interaction']
scaler = StandardScaler()
X_train_prep[numeric_cols] = scaler.fit_transform(X_train_prep[numeric_cols])

# Best parameters found by GridSearchCV in Phase 5
model = RandomForestClassifier(
    n_estimators=200, max_depth=None, min_samples_split=2, random_state=42
)
model.fit(X_train_prep, y_train)

# Save the exact column structure the model expects -- new customers must match this
REFERENCE_COLUMNS = X_train_prep.columns

print("Tuned model ready.")
print(f"Expected columns: {list(REFERENCE_COLUMNS)}")

Tuned model ready.
Expected columns: ['Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Total Spend', 'Last Interaction', 'Subscription Type_Premium', 'Subscription Type_Standard', 'Contract Length_Monthly', 'Contract Length_Quarterly']


## Task 17: Prediction Function

A new customer arrives as a plain dictionary of raw values -- exactly how you'd
naturally describe a real customer, not pre-encoded numbers. This function runs
that dictionary through the *exact same* preprocessing as training (same Gender
map, same one-hot encoding, same fitted scaler), then returns a prediction and a
probability.

In [2]:
def preprocess_customer(customer, reference_columns, scaler, numeric_cols):
    """Turns one raw customer dict into a row matching the model's training format."""
    row = pd.DataFrame([customer])
    row['Gender'] = row['Gender'].map({'Male': 0, 'Female': 1})
    row = pd.get_dummies(row, columns=['Subscription Type', 'Contract Length'], drop_first=True)
    # Add any missing columns (e.g. a category not present in this one customer) as 0,
    # and drop/reorder to match exactly what the model was trained on
    row = row.reindex(columns=reference_columns, fill_value=0)
    row[numeric_cols] = scaler.transform(row[numeric_cols])
    return row


def predict_churn(customer, model, reference_columns, scaler, numeric_cols):
    """Returns a churn prediction and probability for one new customer."""
    row = preprocess_customer(customer, reference_columns, scaler, numeric_cols)
    prediction = model.predict(row)[0]
    probability = model.predict_proba(row)[0][1]  # probability of class 1 (churn)
    return {
        'prediction': 'Churn' if prediction == 1 else 'Stay',
        'churn_probability': round(probability, 4),
    }

In [3]:
# Quick test on a made-up, low-risk-looking customer
test_customer = {
    'Age': 30, 'Gender': 'Male', 'Tenure': 24, 'Usage Frequency': 20,
    'Support Calls': 1, 'Payment Delay': 3, 'Subscription Type': 'Premium',
    'Contract Length': 'Annual', 'Total Spend': 700, 'Last Interaction': 5
}

result = predict_churn(test_customer, model, REFERENCE_COLUMNS, scaler, numeric_cols)
print(result)

{'prediction': 'Stay', 'churn_probability': np.float64(0.0)}


## Task 18: Risk Categorization

We turn the raw probability into a business-friendly label, and -- more
importantly -- identify *why* this specific customer is flagged, using the exact
thresholds Phase 2 and Phase 5 already proved matter most: `Payment Delay` past
15 days, and `Support Calls` at 5 or more.

In [4]:
def categorize_risk(probability):
    if probability >= 0.7:
        return 'High'
    elif probability >= 0.3:
        return 'Medium'
    else:
        return 'Low'


def identify_risk_factors(customer):
    """Checks this specific customer against the known strongest churn drivers."""
    factors = []
    if customer['Payment Delay'] > 15:
        factors.append(f"Payment delay of {customer['Payment Delay']} days (past the 15-day danger threshold)")
    if customer['Support Calls'] >= 5:
        factors.append(f"{customer['Support Calls']} support calls (at/above the 5-call danger threshold)")
    if customer['Contract Length'] == 'Monthly':
        factors.append("On a Monthly contract (higher churn rate than Quarterly/Annual)")
    if customer['Gender'] == 'Female':
        factors.append("Gender shows a higher churn rate in this dataset (secondary factor)")
    if not factors:
        factors.append("No major risk factors detected -- customer profile looks stable")
    return factors

In [5]:
risk_level = categorize_risk(result['churn_probability'])
factors = identify_risk_factors(test_customer)

print(f"Risk level: {risk_level}")
print("Contributing factors:")
for f in factors:
    print(f"  - {f}")

Risk level: Low
Contributing factors:
  - No major risk factors detected -- customer profile looks stable


## Task 19: Recommendation Engine

Based on the risk level and *which specific factor* triggered it, suggest a
concrete next action -- not a generic "reach out to the customer" for everyone.

In [6]:
def recommend_actions(risk_level, factors):
    actions = []

    if risk_level == 'Low':
        actions.append("No action needed -- maintain current experience.")
        return actions

    factor_text = ' '.join(factors)
    if 'Payment delay' in factor_text:
        actions.append("Resolve billing friction: reach out about the payment delay, "
                        "offer a flexible payment plan or reminder system.")
    if 'support calls' in factor_text.lower():
        actions.append("Escalate to a retention specialist: repeated support calls "
                        "signal unresolved frustration -- proactive outreach needed.")
    if 'Monthly contract' in factor_text:
        actions.append("Offer an incentive to switch to a Quarterly or Annual plan "
                        "to increase commitment and lower churn risk.")

    if risk_level == 'High' and not actions:
        actions.append("Model flags high risk without a clear single driver -- "
                        "manually review this customer's full profile.")
    elif risk_level == 'Medium':
        actions.append("Monitor this customer and consider a light engagement nudge "
                        "(check-in email, usage tips) before risk escalates.")

    return actions

In [7]:
actions = recommend_actions(risk_level, factors)
print("Recommended actions:")
for a in actions:
    print(f"  - {a}")

Recommended actions:
  - No action needed -- maintain current experience.


## Putting it all together: one function, full report

This combines Tasks 17-19 into a single call -- feed in a new customer, get back
everything at once.

In [8]:
def generate_customer_report(customer, model, reference_columns, scaler, numeric_cols):
    result = predict_churn(customer, model, reference_columns, scaler, numeric_cols)
    risk_level = categorize_risk(result['churn_probability'])
    factors = identify_risk_factors(customer)
    actions = recommend_actions(risk_level, factors)

    print("=" * 55)
    print(f"Prediction:          {result['prediction']}")
    print(f"Churn probability:   {result['churn_probability']:.1%}")
    print(f"Risk level:          {risk_level}")
    print("Contributing factors:")
    for f in factors:
        print(f"  - {f}")
    print("Recommended actions:")
    for a in actions:
        print(f"  - {a}")
    print("=" * 55)

    return {'prediction': result['prediction'],
            'probability': result['churn_probability'],
            'risk_level': risk_level,
            'factors': factors,
            'actions': actions}

In [9]:
# Demo 1: low-risk customer (long tenure, few calls, pays on time, annual plan)
low_risk_customer = {
    'Age': 30, 'Gender': 'Male', 'Tenure': 24, 'Usage Frequency': 20,
    'Support Calls': 1, 'Payment Delay': 3, 'Subscription Type': 'Premium',
    'Contract Length': 'Annual', 'Total Spend': 700, 'Last Interaction': 5
}
_ = generate_customer_report(low_risk_customer, model, REFERENCE_COLUMNS, scaler, numeric_cols)

Prediction:          Stay
Churn probability:   0.0%
Risk level:          Low
Contributing factors:
  - No major risk factors detected -- customer profile looks stable
Recommended actions:
  - No action needed -- maintain current experience.


In [10]:
# Demo 2: high-risk customer (long payment delay AND many support calls, monthly plan)
high_risk_customer = {
    'Age': 35, 'Gender': 'Female', 'Tenure': 5, 'Usage Frequency': 8,
    'Support Calls': 7, 'Payment Delay': 22, 'Subscription Type': 'Basic',
    'Contract Length': 'Monthly', 'Total Spend': 250, 'Last Interaction': 25
}
_ = generate_customer_report(high_risk_customer, model, REFERENCE_COLUMNS, scaler, numeric_cols)

Prediction:          Churn
Churn probability:   97.5%
Risk level:          High
Contributing factors:
  - Payment delay of 22 days (past the 15-day danger threshold)
  - 7 support calls (at/above the 5-call danger threshold)
  - On a Monthly contract (higher churn rate than Quarterly/Annual)
  - Gender shows a higher churn rate in this dataset (secondary factor)
Recommended actions:
  - Resolve billing friction: reach out about the payment delay, offer a flexible payment plan or reminder system.
  - Escalate to a retention specialist: repeated support calls signal unresolved frustration -- proactive outreach needed.
  - Offer an incentive to switch to a Quarterly or Annual plan to increase commitment and lower churn risk.


**An honest observation, testing this system:** because Phase 5 showed this
model fits the data almost perfectly (~99.8% F1), its probability outputs tend to
be very confident -- close to 0% or close to 100% -- rather than landing in a
genuine middle ground. I tested dozens of customer profiles right around the
15-day / 5-call thresholds while building this, and the model rarely returned
anything in the 30-70% "Medium" range; it snaps to one side almost as soon as a
threshold is crossed. The `Medium` category is still implemented and would
trigger normally on messier, real-world data -- it's just rare *with this
particular dataset and model*. That's worth noting as a real limitation in your
Task 21 write-up: this system is extremely confident, but that confidence comes
from unusually clean patterns in the training data, not necessarily from how
messy real customer behavior actually is.

---
### ✅ Phase 6 checkpoint

You now have a complete, working system:
- `predict_churn()` -- raw customer info in, prediction + probability out
- `categorize_risk()` + `identify_risk_factors()` -- turns probability into a
  labeled risk level with named reasons
- `recommend_actions()` -- turns risk + reasons into concrete next steps
- `generate_customer_report()` -- the single function tying all of it together

**Next: Phase 7 (Tasks 20-21)** -- the final step. Pulling everything from all
six phases into one overall analysis and a written conclusion.